# 02 · Preprocessing Validation

Validates each preprocessing stage and produces diagnostic figures for
**Section 3 (Dataset & Preprocessing)** of the paper.

Checks:
1. Stage A — column renames and type coercions
2. Stage B — partition completeness and gap statistics
3. Stage C — truncation fractions
4. Stage D — fragmentation distribution shape

Paper diagnostics:
- Distribution of pods per hour over 185 days (anomalous days?)
- Distribution of `gpu_util` values (bimodal? long-tailed? excess zeros?)
- Fraction of pods left- and right-truncated

In [ ]:
import json
import duckdb
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

from src.config import (
    INTERIM_DIR, PROCESSED_DIR, LOG_DIR,
    DUCKDB_MEMORY_LIMIT, DUCKDB_THREADS,
    TRACE_END_HOUR, TRACE_DAYS,
)
from src.viz.style import apply_ieee_style, figure_single, figure_double
apply_ieee_style()

con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET threads={DUCKDB_THREADS}")
print("Ready")

## Stage A — Canonicalization

In [ ]:
pod_src = INTERIM_DIR / 'pod_hourly_canonical.parquet'
srv_src = INTERIM_DIR / 'server_hourly_canonical.parquet'

print('=== pod_hourly_canonical ===')
pod_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{pod_src}')").df()
display(pod_df)

n_pod = con.execute(f"SELECT count(*) FROM read_parquet('{pod_src}')").fetchone()[0]
n_pods_unique = con.execute(f"SELECT count(DISTINCT pod_uid) FROM read_parquet('{pod_src}')").fetchone()[0]
print(f"Rows: {n_pod:,}  |  Unique pods: {n_pods_unique:,}")

# State distribution
states = con.execute(
    f"SELECT state, count(*) AS n FROM read_parquet('{pod_src}') GROUP BY state ORDER BY n DESC"
).df()
print('\nState distribution:')
display(states)

In [ ]:
# gpu_util range sanity check — should be 0.0 to 1.0
util_stats = con.execute(f"""
    SELECT
        min(gpu_util)  AS util_min,
        max(gpu_util)  AS util_max,
        avg(gpu_util)  AS util_mean,
        count(*) FILTER (WHERE gpu_util IS NULL) AS null_count,
        count(*) FILTER (WHERE gpu_util > 1.0)   AS above_1_count
    FROM read_parquet('{pod_src}')
""").df()
print('gpu_util stats:')
display(util_stats)

## Stage B — Completeness

In [ ]:
report_path = LOG_DIR / 'completeness_report.json'
with open(report_path) as f:
    completeness = json.load(f)

print(json.dumps(completeness, indent=2))

In [ ]:
mask_src = INTERIM_DIR / 'pod_activity_mask.parquet'

# ── Pods per hour over trace duration ────────────────────────────────────────
pods_per_hour = con.execute(f"""
    SELECT rel_hour, count(*) AS pod_count
    FROM read_parquet('{mask_src}')
    WHERE is_present
    GROUP BY rel_hour
    ORDER BY rel_hour
""").df()

fig, ax = figure_double(height_ratio=0.7)
ax.plot(pods_per_hour['rel_hour'] / 24, pods_per_hour['pod_count'], lw=0.8)
ax.set_xlabel('Trace day')
ax.set_ylabel('Active pods')
ax.set_title('Active pod count per hour')
ax.xaxis.set_major_locator(mticker.MultipleLocator(20))
plt.tight_layout()
plt.show()

# Anomalous days: hours with pod count below 5th percentile
p5 = np.percentile(pods_per_hour['pod_count'], 5)
anomalous = pods_per_hour[pods_per_hour['pod_count'] < p5]
print(f'Anomalous hours (below p5={p5:.0f}): {len(anomalous)}')
display(anomalous.head(10))

## Stage C — Truncation

In [ ]:
trunc_src = INTERIM_DIR / 'pod_truncation_flags.parquet'

trunc_summary = con.execute(f"""
    SELECT
        count(*)                         AS total_pods,
        sum(left_truncated::INT)         AS left_trunc,
        sum(right_truncated::INT)        AS right_trunc,
        sum((left_truncated OR right_truncated)::INT) AS either_trunc,
        round(sum(left_truncated::INT)  * 100.0 / count(*), 2) AS pct_left,
        round(sum(right_truncated::INT) * 100.0 / count(*), 2) AS pct_right
    FROM read_parquet('{trunc_src}')
""").df()
display(trunc_summary)

# Terminal state distribution
terminal = con.execute(f"""
    SELECT terminal_state, count(*) AS n
    FROM read_parquet('{trunc_src}')
    GROUP BY terminal_state ORDER BY n DESC
""").df()
display(terminal)

## gpu_util distribution (paper: Section 3)

In [ ]:
# Sample 200K rows for the histogram (avoid loading everything into RAM)
util_sample = con.execute(f"""
    SELECT gpu_util
    FROM read_parquet('{pod_src}')
    WHERE state = 'RUNNING'
      AND gpu_util IS NOT NULL
    USING SAMPLE 200000
""").df()

fig, axes = plt.subplots(1, 2, figsize=(7, 2.4))

axes[0].hist(util_sample['gpu_util'], bins=50, edgecolor='none', color='#0072B2')
axes[0].set_xlabel('gpu_util')
axes[0].set_ylabel('Count (sample)')
axes[0].set_title('GPU utilization distribution')

axes[1].hist(util_sample['gpu_util'], bins=50, edgecolor='none', color='#0072B2',
             density=True, cumulative=True)
axes[1].set_xlabel('gpu_util')
axes[1].set_ylabel('CDF')
axes[1].set_title('CDF')

plt.tight_layout()
plt.show()

frac_zero = (util_sample['gpu_util'] == 0).mean()
frac_full = (util_sample['gpu_util'] >= 0.99).mean()
print(f'Fraction util=0:   {frac_zero:.2%}')
print(f'Fraction util>=99: {frac_full:.2%}')

## Stage D — Fragmentation aggregates

In [ ]:
frag_src = PROCESSED_DIR / 'hourly_frag.parquet'

frag_summary = con.execute(f"""
    SELECT
        server_tier,
        count(DISTINCT rel_hour)          AS hours_present,
        sum(total_gpus_requested)         AS total_requested,
        sum(total_gpus_used)              AS total_used,
        sum(total_frag_gpu_hours)         AS total_frag,
        round(sum(total_frag_gpu_hours) * 100.0 / sum(total_gpus_requested), 2) AS frag_pct
    FROM read_parquet('{frag_src}')
    GROUP BY server_tier
    ORDER BY server_tier
""").df()
print('Fragmentation by server tier:')
display(frag_summary)

# Hourly frag time series
hourly = con.execute(f"""
    SELECT rel_hour, sum(total_frag_gpu_hours) AS frag_total
    FROM read_parquet('{frag_src}')
    GROUP BY rel_hour ORDER BY rel_hour
""").df()

fig, ax = figure_double(height_ratio=0.7)
ax.plot(hourly['rel_hour'] / 24, hourly['frag_total'], lw=0.7, color='#D55E00')
ax.set_xlabel('Trace day')
ax.set_ylabel('Fragmented GPU-hours')
ax.set_title('Hourly GPU fragmentation')
plt.tight_layout()
plt.show()

## Phase 2 — Real v2026 Data Validation

Validates outputs of the rewritten preprocessing pipeline against the three
confirmed tables: `server_hourly` (143M rows), `job_execution_summary` (40M rows),
`network_hourly` (5M rows).

Run `make preprocess` before executing these cells.

In [ ]:
import json
from src.config import table_glob

# ── Fleet capacity over time ──────────────────────────────────────────────────
fleet_src = str(PROCESSED_DIR / 'hourly_fleet_capacity.parquet').replace('\\', '/')
fleet_df = con.execute(f"""
    SELECT day,
           sum(total_gpus_available) / count(*) AS mean_gpus,
           max(total_gpus_available)             AS max_gpus
    FROM read_parquet('{fleet_src}')
    GROUP BY day ORDER BY day
""").df()

fig, ax = figure_double(height_ratio=0.7)
ax.plot(fleet_df['day'], fleet_df['mean_gpus'] / 1000, lw=0.8, color='#0072B2', label='mean')
ax.fill_between(fleet_df['day'],
                fleet_df['mean_gpus'] / 1000,
                fleet_df['max_gpus']  / 1000,
                alpha=0.2, color='#0072B2', label='max')
ax.set_xlabel('Trace day')
ax.set_ylabel('GPUs available (K)')
ax.set_title('Fleet GPU capacity over 185 days')
ax.legend()
plt.tight_layout(); plt.show()
print(f"Mean fleet size: {fleet_df['mean_gpus'].mean():,.0f} GPUs  |  "
      f"Max: {fleet_df['max_gpus'].max():,.0f}")

# ── Job type distribution ─────────────────────────────────────────────────────
jts_src = str(PROCESSED_DIR / 'job_type_summary.parquet').replace('\\', '/')
job_type_df = con.execute(f"""
    SELECT job_type_public,
           sum(job_count)               AS total_jobs,
           sum(total_gpu_hours_requested) AS total_gpu_hours
    FROM read_parquet('{jts_src}')
    GROUP BY job_type_public
    ORDER BY total_jobs DESC
""").df()
print('\n=== Job type distribution ===')
display(job_type_df)

fig, axes = plt.subplots(1, 2, figsize=(7, 2.5))
axes[0].barh(job_type_df['job_type_public'], job_type_df['total_jobs'] / 1e6,
             color='#0072B2')
axes[0].set_xlabel('Jobs (M)')
axes[0].set_title('Jobs by type')

axes[1].barh(job_type_df['job_type_public'], job_type_df['total_gpu_hours'] / 1e6,
             color='#E69F00')
axes[1].set_xlabel('GPU-hours requested (M)')
axes[1].set_title('GPU-hours by type')
plt.tight_layout(); plt.show()

# ── GPU spec distribution ─────────────────────────────────────────────────────
spec_df = con.execute(f"""
    SELECT gpu_spec_public,
           sum(job_count) AS jobs
    FROM read_parquet('{jts_src}')
    WHERE gpu_spec_public IS NOT NULL
    GROUP BY gpu_spec_public
    ORDER BY jobs DESC
    LIMIT 12
""").df()
print('\n=== GPU spec distribution (top 12) ===')
display(spec_df)

# ── Priority class split ──────────────────────────────────────────────────────
prio_df = con.execute(f"""
    SELECT priority_class,
           sum(job_count)               AS total_jobs,
           sum(total_gpu_hours_requested) AS total_gpu_hours
    FROM read_parquet('{jts_src}')
    GROUP BY priority_class ORDER BY total_jobs DESC
""").df()
print('\n=== Priority class split ===')
display(prio_df)

# ── gpu_request CDF (fractional GPU sharing) ──────────────────────────────────
job_src = str(INTERIM_DIR / 'job_summary_canonical.parquet').replace('\\', '/')
gpu_req = con.execute(f"""
    SELECT gpu_request
    FROM read_parquet('{job_src}')
    USING SAMPLE 200000
""").df()

vals = gpu_req['gpu_request'].sort_values().dropna()
cdf  = [i / len(vals) for i in range(1, len(vals) + 1)]

fig, axes = plt.subplots(1, 2, figsize=(7, 2.5))
axes[0].plot(vals.values, cdf, lw=0.8, color='#0072B2')
axes[0].set_xlabel('gpu_request (GPUs)')
axes[0].set_ylabel('CDF')
axes[0].set_title('GPU request CDF (fractional sharing)')

axes[1].hist(vals[vals <= 1.0].values, bins=50, edgecolor='none', color='#009E73')
axes[1].set_xlabel('gpu_request (≤ 1.0 GPU)')
axes[1].set_ylabel('Count (sample)')
axes[1].set_title('Sub-1-GPU requests')
plt.tight_layout(); plt.show()

frac_frac = (vals <= 1.0).mean()
print(f'Fraction of jobs requesting < 1 full GPU: {frac_frac:.1%}')

# ── Allocation efficiency histogram ───────────────────────────────────────────
eff_src = str(PROCESSED_DIR / 'server_allocation_efficiency.parquet').replace('\\', '/')
eff_df = con.execute(f"""
    SELECT allocation_efficiency, server_tier
    FROM read_parquet('{eff_src}')
    WHERE allocation_efficiency IS NOT NULL
""").df()

fig, ax = figure_single(height_ratio=1.1)
for tier, color in [('small','#56B4E9'), ('medium','#0072B2'), ('large','#D55E00')]:
    sub = eff_df[eff_df['server_tier'] == tier]['allocation_efficiency']
    if len(sub):
        ax.hist(sub.clip(0, 2), bins=40, alpha=0.6, label=tier, color=color, density=True)
ax.axvline(1.0, color='black', lw=0.8, ls='--', label='100% utilised')
ax.set_xlabel('Allocation efficiency (requested / capacity)')
ax.set_ylabel('Density')
ax.set_title('Server allocation efficiency')
ax.legend()
plt.tight_layout(); plt.show()

over = (eff_df['allocation_efficiency'] > 1.0).mean()
under = (eff_df['allocation_efficiency'] < 0.5).mean()
print(f'Oversubscribed servers (>1): {over:.1%}')
print(f'Under-utilised servers (<50%): {under:.1%}')

# ── TDP coverage ──────────────────────────────────────────────────────────────
tdp_path = LOG_DIR / f'tdp_coverage_v2026.json'
if tdp_path.exists():
    with open(tdp_path) as f:
        tdp = json.load(f)
    print(f'\n=== TDP coverage ===')
    print(f'Distinct GPU specs: {tdp["distinct_specs"]}')
    print(f'Fallback (unknown): {tdp["fallback_count"]}  {tdp["fallback_specs"]}')
    import pandas as pd
    display(pd.DataFrame(tdp['coverage']))